In [ ]:
"""
Dynamic Pathfinding Agent — Tkinter GUI
========================================
Run from Jupyter Notebook with:
    %run pathfinding.py

Or run directly:
    python pathfinding.py

No extra pip installs needed — tkinter is built into Python.
"""

# ==============================================================
# IMPORTS
# ==============================================================
import tkinter as tk
from tkinter import ttk, messagebox
import random
import math
import time
import heapq
from collections import defaultdict

# ==============================================================
# CONSTANTS — Cell Types
# ==============================================================
EMPTY    = 0
WALL     = 1
START    = 2
GOAL     = 3
VISITED  = 4   # expanded nodes  → Blue
FRONTIER = 5   # in open queue   → Yellow
PATH     = 6   # final path      → Green
DYN_WALL = 7   # dynamically spawned obstacle

COLORS = {
    EMPTY:    "#1e1e2e",
    WALL:     "#585b70",
    START:    "#a6e3a1",
    GOAL:     "#f38ba8",
    VISITED:  "#89b4fa",
    FRONTIER: "#f9e2af",
    PATH:     "#a6e3a1",
    DYN_WALL: "#d45555",
}

CELL_SIZE   = 28          # pixels per grid cell
GRID_ROWS   = 20
GRID_COLS   = 20
ANIM_DELAY  = 30          # ms between animation frames
AGENT_DELAY = 120         # ms between agent steps in dynamic mode


# ==============================================================
# HEURISTICS
# ==============================================================
def manhattan(a, b):
    """Manhattan Distance  –  |Δrow| + |Δcol|"""
    return abs(a[0] - b[0]) + abs(a[1] - b[1])

def euclidean(a, b):
    """Euclidean Distance  –  √(Δrow² + Δcol²)"""
    return math.sqrt((a[0] - b[0])**2 + (a[1] - b[1])**2)


# ==============================================================
# PRIORITY QUEUE (Min-Heap wrapper)
# ==============================================================
class PriorityQueue:
    def __init__(self):
        self._heap = []
        self._counter = 0          # tie-breaker

    def push(self, priority, item):
        heapq.heappush(self._heap, (priority, self._counter, item))
        self._counter += 1

    def pop(self):
        _, _, item = heapq.heappop(self._heap)
        return item

    def __len__(self):
        return len(self._heap)

    def empty(self):
        return len(self._heap) == 0


# ==============================================================
# SEARCH RESULT
# ==============================================================
class SearchResult:
    def __init__(self):
        self.path          = []
        self.visited_count = 0
        self.path_cost     = 0
        self.exec_ms       = 0.0
        self.success       = False


# ==============================================================
# GRID ENVIRONMENT
# ==============================================================
class Grid:
    """
    Manages the 2-D grid world.
    Responsibilities:
      - Cell state storage
      - Wall placement / removal
      - Start & Goal positioning
      - Random map generation
      - Dynamic obstacle spawning
      - Neighbour queries for search
    """

    def __init__(self, rows=GRID_ROWS, cols=GRID_COLS):
        self.rows  = rows
        self.cols  = cols
        self.cells = [[EMPTY] * cols for _ in range(rows)]
        self.start = (1, 1)
        self.goal  = (rows - 2, cols - 2)
        self._mark_endpoints()

    # ---- Internal ----
    def _mark_endpoints(self):
        self.cells[self.start[0]][self.start[1]] = START
        self.cells[self.goal[0]][self.goal[1]]   = GOAL

    # ---- Reset ----
    def reset(self, rows=None, cols=None):
        if rows: self.rows = rows
        if cols: self.cols = cols
        self.cells = [[EMPTY] * self.cols for _ in range(self.rows)]
        # clamp start/goal
        self.start = (min(self.start[0], self.rows - 2), min(self.start[1], self.cols - 2))
        self.goal  = (min(self.goal[0],  self.rows - 2), min(self.goal[1],  self.cols - 2))
        self._mark_endpoints()

    def clear_visuals(self):
        """Remove VISITED / FRONTIER / PATH markers only."""
        for r in range(self.rows):
            for c in range(self.cols):
                if self.cells[r][c] in (VISITED, FRONTIER, PATH):
                    self.cells[r][c] = EMPTY

    # ---- Wall editing ----
    def toggle_wall(self, r, c):
        if (r, c) in (self.start, self.goal): return
        if not (0 <= r < self.rows and 0 <= c < self.cols): return
        self.cells[r][c] = EMPTY if self.cells[r][c] in (WALL, DYN_WALL) else WALL

    def set_wall(self, r, c):
        if (r, c) in (self.start, self.goal): return
        if 0 <= r < self.rows and 0 <= c < self.cols:
            self.cells[r][c] = WALL

    def remove_wall(self, r, c):
        if 0 <= r < self.rows and 0 <= c < self.cols:
            if self.cells[r][c] in (WALL, DYN_WALL):
                self.cells[r][c] = EMPTY

    # ---- Start / Goal ----
    def move_start(self, r, c):
        if not (0 <= r < self.rows and 0 <= c < self.cols): return
        if self.cells[r][c] in (WALL, DYN_WALL) or (r, c) == self.goal: return
        self.cells[self.start[0]][self.start[1]] = EMPTY
        self.start = (r, c)
        self.cells[r][c] = START

    def move_goal(self, r, c):
        if not (0 <= r < self.rows and 0 <= c < self.cols): return
        if self.cells[r][c] in (WALL, DYN_WALL) or (r, c) == self.start: return
        self.cells[self.goal[0]][self.goal[1]] = EMPTY
        self.goal = (r, c)
        self.cells[r][c] = GOAL

    # ---- Random map ----
    def generate_random(self, density=0.30):
        """Fill grid with random walls at the given density (0–1)."""
        self.cells = [[EMPTY] * self.cols for _ in range(self.rows)]
        for r in range(self.rows):
            for c in range(self.cols):
                if (r, c) in (self.start, self.goal): continue
                if random.random() < density:
                    self.cells[r][c] = WALL
        self._mark_endpoints()

    # ---- Dynamic obstacle spawning ----
    def spawn_obstacle(self, probability, protected=None):
        """
        Attempt to place one new dynamic obstacle.
        Returns (row, col) if spawned, None otherwise.
        Only recalculates if the obstacle is ON the active path.
        """
        if random.random() > probability:
            return None
        candidates = [
            (r, c)
            for r in range(self.rows)
            for c in range(self.cols)
            if self.cells[r][c] == EMPTY
            and (r, c) != self.start
            and (r, c) != self.goal
            and (protected is None or (r, c) not in protected)
        ]
        if not candidates:
            return None
        cell = random.choice(candidates)
        self.cells[cell[0]][cell[1]] = DYN_WALL
        return cell

    # ---- Neighbour query ----
    def walkable(self, r, c):
        return (0 <= r < self.rows and 0 <= c < self.cols
                and self.cells[r][c] not in (WALL, DYN_WALL))

    def get_neighbors(self, r, c):
        for dr, dc in ((-1,0),(1,0),(0,-1),(0,1)):
            nr, nc = r+dr, c+dc
            if self.walkable(nr, nc):
                yield (nr, nc)


# ==============================================================
# SEARCH ALGORITHMS — step-by-step generators
# ==============================================================
def search_step_generator(grid: Grid, algorithm: str, heuristic_fn,
                           start_override=None):
    """
    Generator that runs A* or GBFS one expansion at a time.

    Yields dicts:
      {'type': 'expand', 'cell': (r,c), 'frontier': set, 'visited': set}
    Final yield:
      {'type': 'done', 'result': SearchResult}
    """
    start_pos = start_override if start_override else grid.start
    goal_pos  = grid.goal

    open_q  = PriorityQueue()
    g_score = {start_pos: 0}
    came_from = {}
    closed  = set()
    in_open = {start_pos}

    h0 = heuristic_fn(start_pos, goal_pos)
    f0 = (g_score[start_pos] + h0) if algorithm == "A*" else h0
    open_q.push(f0, start_pos)

    t_start = time.perf_counter()
    result  = SearchResult()

    while not open_q.empty():
        current = open_q.pop()
        in_open.discard(current)

        if current in closed:
            continue
        closed.add(current)

        # Mark VISITED on the grid
        if current != start_pos and current != goal_pos:
            grid.cells[current[0]][current[1]] = VISITED

        # ── Goal check ──
        if current == goal_pos:
            # Reconstruct path
            path = [current]
            node = current
            while node in came_from:
                node = came_from[node]
                path.append(node)
            path.reverse()
            # Paint PATH
            for cell in path:
                if cell != start_pos and cell != goal_pos:
                    grid.cells[cell[0]][cell[1]] = PATH

            result.path          = path
            result.visited_count = len(closed)
            result.path_cost     = g_score.get(goal_pos, len(path) - 1)
            result.exec_ms       = (time.perf_counter() - t_start) * 1000
            result.success       = True
            yield {'type': 'done', 'result': result}
            return

        # ── Expand neighbours ──
        for neighbour in grid.get_neighbors(current[0], current[1]):
            if neighbour in closed:
                continue
            tentative_g = g_score[current] + 1
            if tentative_g < g_score.get(neighbour, float('inf')):
                came_from[neighbour] = current
                g_score[neighbour]   = tentative_g
                h = heuristic_fn(neighbour, goal_pos)
                f = (tentative_g + h) if algorithm == "A*" else h
                open_q.push(f, neighbour)
                in_open.add(neighbour)
                if neighbour != start_pos and neighbour != goal_pos:
                    grid.cells[neighbour[0]][neighbour[1]] = FRONTIER

        result.visited_count = len(closed)
        result.exec_ms       = (time.perf_counter() - t_start) * 1000
        yield {'type': 'expand', 'result': result}

    # No path
    result.exec_ms   = (time.perf_counter() - t_start) * 1000
    result.success   = False
    yield {'type': 'done', 'result': result}


def run_instant_search(grid: Grid, algorithm: str, heuristic_fn,
                        start_override=None) -> SearchResult:
    """Run search to completion without animation (used for re-planning)."""
    gen = search_step_generator(grid, algorithm, heuristic_fn, start_override)
    result = SearchResult()
    for frame in gen:
        result = frame['result']
        if frame['type'] == 'done':
            break
    return result


# ==============================================================
# MAIN APPLICATION
# ==============================================================
class PathfindingApp:
    """
    Top-level application class.
    Manages:
      - Tkinter root window
      - Canvas grid rendering
      - Control panel (menu bar top area)
      - Event handling (mouse, keyboard)
      - Search animation loop
      - Dynamic mode loop
    """

    def __init__(self):
        self.root = tk.Tk()
        self.root.title("Dynamic Pathfinding Agent")
        self.root.configure(bg="#1e1e2e")
        self.root.resizable(True, True)

        # Grid
        self.grid       = Grid(GRID_ROWS, GRID_COLS)
        self.cell_size  = CELL_SIZE

        # Algorithm & heuristic state
        self.algorithm     = tk.StringVar(value="A*")
        self.heuristic_var = tk.StringVar(value="Manhattan")
        self.dynamic_var   = tk.BooleanVar(value=False)
        self.density_var   = tk.StringVar(value="30")
        self.rows_var      = tk.StringVar(value=str(GRID_ROWS))
        self.cols_var      = tk.StringVar(value=str(GRID_COLS))
        self.speed_var     = tk.IntVar(value=5)  # steps per frame

        # Search state
        self.search_gen    = None
        self.searching     = False
        self.search_result = SearchResult()
        self.search_job    = None   # after() handle

        # Dynamic agent state
        self.agent_pos    = None
        self.agent_moving = False
        self.active_path  = []
        self.path_idx     = 0
        self.replan_count = 0
        self.agent_job    = None

        # Mouse interaction
        self.draw_mode        = "wall"   # 'wall' | 'erase' | 'start' | 'goal'
        self.last_drawn_cell  = None

        # Build UI
        self._build_ui()
        self._bind_keys()
        self._redraw()

    # ============================================================
    # UI CONSTRUCTION
    # ============================================================
    def _build_ui(self):
        """Create all widgets."""

        # ── Top bar: keyboard shortcut reference ──
        top_frame = tk.Frame(self.root, bg="#181825", bd=0)
        top_frame.pack(fill="x", padx=6, pady=(6, 0))

        tk.Label(
            top_frame, text="Keyboard Shortcuts:",
            bg="#181825", fg="#89b4fa",
            font=("Courier", 10, "bold")
        ).pack(side="left", padx=(8, 12))

        shortcuts = [
            ("S", "Start"),
            ("A", "A*"),
            ("G", "GBFS"),
            ("M", "Manhattan"),
            ("E", "Euclidean"),
            ("D", "Dynamic"),
            ("R", "Random Map"),
            ("C", "Clear"),
            ("X", "Reset"),
            ("1", "Place Start"),
            ("2", "Place Goal"),
        ]
        for key, label in shortcuts:
            tk.Label(
                top_frame,
                text=f"[{key}] {label}",
                bg="#181825", fg="#cdd6f4",
                font=("Courier", 9), padx=4
            ).pack(side="left")

        # ── Controls row ──
        ctrl = tk.Frame(self.root, bg="#1e1e2e")
        ctrl.pack(fill="x", padx=6, pady=4)

        style_btn = {"bg": "#313244", "fg": "#cdd6f4", "font": ("Courier", 10),
                     "relief": "flat", "padx": 8, "pady": 3,
                     "activebackground": "#45475a", "cursor": "hand2"}
        style_lbl = {"bg": "#1e1e2e", "fg": "#a6adc8", "font": ("Courier", 10)}
        style_ent = {"bg": "#313244", "fg": "#cdd6f4", "font": ("Courier", 10),
                     "relief": "flat", "insertbackground": "#cdd6f4", "width": 4}
        style_dd  = {"background": "#313244", "foreground": "#cdd6f4",
                     "font": ("Courier", 10)}

        # Algorithm selector
        tk.Label(ctrl, text="Algo:", **style_lbl).pack(side="left", padx=(0,2))
        algo_menu = ttk.Combobox(ctrl, textvariable=self.algorithm,
                                 values=["A*", "GBFS"], state="readonly", width=6)
        algo_menu.pack(side="left", padx=(0,8))

        # Heuristic selector
        tk.Label(ctrl, text="Heuristic:", **style_lbl).pack(side="left", padx=(0,2))
        heur_menu = ttk.Combobox(ctrl, textvariable=self.heuristic_var,
                                 values=["Manhattan", "Euclidean"], state="readonly", width=10)
        heur_menu.pack(side="left", padx=(0,8))

        # Dynamic toggle
        tk.Checkbutton(
            ctrl, text="Dynamic Mode", variable=self.dynamic_var,
            bg="#1e1e2e", fg="#a6e3a1", selectcolor="#1e1e2e",
            activebackground="#1e1e2e", font=("Courier", 10),
            command=self._on_dynamic_toggle
        ).pack(side="left", padx=(0,8))

        # Buttons
        tk.Button(ctrl, text="▶ Search",     command=self.start_search,  **style_btn).pack(side="left", padx=2)
        tk.Button(ctrl, text="↺ Clear",      command=self.clear_search,  **style_btn).pack(side="left", padx=2)
        tk.Button(ctrl, text="✕ Reset",      command=self.clear_all,     **style_btn).pack(side="left", padx=2)
        tk.Button(ctrl, text="Random Map",   command=self.random_map,    **style_btn).pack(side="left", padx=2)
        tk.Button(ctrl, text="Set Start [1]",command=lambda: self._set_draw_mode("start"), **style_btn).pack(side="left", padx=2)
        tk.Button(ctrl, text="Set Goal [2]", command=lambda: self._set_draw_mode("goal"),  **style_btn).pack(side="left", padx=2)

        # Grid size
        tk.Label(ctrl, text="  Rows:", **style_lbl).pack(side="left")
        tk.Entry(ctrl, textvariable=self.rows_var, **style_ent).pack(side="left", padx=(0,2))
        tk.Label(ctrl, text="Cols:", **style_lbl).pack(side="left")
        tk.Entry(ctrl, textvariable=self.cols_var, **style_ent).pack(side="left", padx=(0,2))
        tk.Button(ctrl, text="Apply Size", command=self.apply_size, **style_btn).pack(side="left", padx=2)

        # Density
        tk.Label(ctrl, text="  Density%:", **style_lbl).pack(side="left")
        tk.Entry(ctrl, textvariable=self.density_var, **style_ent).pack(side="left", padx=(0,8))

        # ── Metrics bar ──
        metrics_frame = tk.Frame(self.root, bg="#181825", bd=0)
        metrics_frame.pack(fill="x", padx=6, pady=(0, 4))

        self.lbl_status   = self._metric_label(metrics_frame, "Status:", "Idle")
        self.lbl_visited  = self._metric_label(metrics_frame, "Nodes Visited:", "0")
        self.lbl_cost     = self._metric_label(metrics_frame, "Path Cost:", "0")
        self.lbl_time     = self._metric_label(metrics_frame, "Exec Time:", "0.00 ms")
        self.lbl_replans  = self._metric_label(metrics_frame, "Re-plans:", "0")

        # ── Canvas ──
        canvas_frame = tk.Frame(self.root, bg="#1e1e2e")
        canvas_frame.pack(padx=6, pady=(0, 2))

        cw = self.grid.cols * self.cell_size
        ch = self.grid.rows * self.cell_size
        self.canvas = tk.Canvas(
            canvas_frame, width=cw, height=ch,
            bg=COLORS[EMPTY], highlightthickness=1,
            highlightbackground="#313244"
        )
        self.canvas.pack()

        # Canvas mouse bindings
        self.canvas.bind("<ButtonPress-1>",   self._on_click)
        self.canvas.bind("<B1-Motion>",       self._on_drag)
        self.canvas.bind("<ButtonRelease-1>", lambda e: setattr(self, 'last_drawn_cell', None))
        self.canvas.bind("<Button-3>",        self._on_right_click)

        # ── Legend ──
        legend_frame = tk.Frame(self.root, bg="#1e1e2e")
        legend_frame.pack(fill="x", padx=6, pady=(0, 6))

        legend_items = [
            ("#a6e3a1", "Start / Path"),
            ("#f38ba8", "Goal"),
            ("#585b70", "Wall"),
            ("#f9e2af", "Frontier"),
            ("#89b4fa", "Visited"),
            ("#d45555", "Dynamic Wall"),
            ("#fab387", "Agent"),
        ]
        for color, label in legend_items:
            f = tk.Frame(legend_frame, bg="#1e1e2e")
            f.pack(side="left", padx=6)
            tk.Label(f, bg=color, width=2, height=1, relief="flat").pack(side="left", padx=(0,3))
            tk.Label(f, text=label, bg="#1e1e2e", fg="#a6adc8",
                     font=("Courier", 9)).pack(side="left")

    def _metric_label(self, parent, label_text, init_val):
        """Create a label+value pair in the metrics bar. Returns the value label."""
        f = tk.Frame(parent, bg="#181825")
        f.pack(side="left", padx=10, pady=4)
        tk.Label(f, text=label_text, bg="#181825", fg="#a6adc8",
                 font=("Courier", 10)).pack(side="left")
        val = tk.Label(f, text=init_val, bg="#181825", fg="#a6e3a1",
                       font=("Courier", 10, "bold"))
        val.pack(side="left", padx=(4, 0))
        return val

    # ============================================================
    # KEYBOARD BINDINGS
    # ============================================================
    def _bind_keys(self):
        r = self.root
        r.bind("<s>", lambda e: self.start_search())
        r.bind("<a>", lambda e: self.algorithm.set("A*"))
        r.bind("<g>", lambda e: self.algorithm.set("GBFS"))
        r.bind("<m>", lambda e: self.heuristic_var.set("Manhattan"))
        r.bind("<e>", lambda e: self.heuristic_var.set("Euclidean"))
        r.bind("<d>", lambda e: self.dynamic_var.set(not self.dynamic_var.get()))
        r.bind("<r>", lambda e: self.random_map())
        r.bind("<c>", lambda e: self.clear_search())
        r.bind("<x>", lambda e: self.clear_all())
        r.bind("<1>", lambda e: self._set_draw_mode("start"))
        r.bind("<2>", lambda e: self._set_draw_mode("goal"))

    # ============================================================
    # DRAWING MODE
    # ============================================================
    def _set_draw_mode(self, mode):
        self.draw_mode = mode
        self._update_status(f"Click grid to place {mode.capitalize()}")

    # ============================================================
    # CANVAS RENDERING
    # ============================================================
    def _redraw(self):
        """Redraw the entire canvas from the grid model."""
        self.canvas.delete("all")
        cs = self.cell_size
        g  = self.grid

        for r in range(g.rows):
            for c in range(g.cols):
                x1, y1 = c * cs, r * cs
                x2, y2 = x1 + cs, y1 + cs
                color = COLORS.get(g.cells[r][c], COLORS[EMPTY])
                self.canvas.create_rectangle(
                    x1, y1, x2, y2,
                    fill=color, outline="#313244", width=1
                )

        # Labels on Start and Goal
        sx, sy = g.start[1]*cs + cs//2, g.start[0]*cs + cs//2
        self.canvas.create_text(sx, sy, text="S", fill="#1e1e2e",
                                font=("Courier", max(8, cs//2), "bold"))
        gx, gy = g.goal[1]*cs + cs//2, g.goal[0]*cs + cs//2
        self.canvas.create_text(gx, gy, text="G", fill="#1e1e2e",
                                font=("Courier", max(8, cs//2), "bold"))

        # Agent dot (dynamic mode)
        if self.agent_pos:
            r0, c0 = self.agent_pos
            ax = c0*cs + cs//2
            ay = r0*cs + cs//2
            rad = max(cs//3, 5)
            self.canvas.create_oval(
                ax-rad, ay-rad, ax+rad, ay+rad,
                fill="#fab387", outline="#ffffff", width=2
            )

    # ============================================================
    # METRIC UPDATES
    # ============================================================
    def _update_metrics(self):
        r = self.search_result
        self.lbl_visited.config(text=str(r.visited_count))
        self.lbl_cost.config(text=str(r.path_cost))
        self.lbl_time.config(text=f"{r.exec_ms:.2f} ms")
        self.lbl_replans.config(text=str(self.replan_count))

    def _update_status(self, msg, color="#f9e2af"):
        self.lbl_status.config(text=msg, fg=color)

    # ============================================================
    # BUTTON / KEY CALLBACKS
    # ============================================================
    def _on_dynamic_toggle(self):
        pass  # variable tracks state; no extra work needed here

    def apply_size(self):
        self._stop_all()
        try:
            r = max(5, min(80, int(self.rows_var.get())))
            c = max(5, min(80, int(self.cols_var.get())))
        except ValueError:
            return
        self.grid.reset(r, c)
        # Resize canvas
        cs = self.cell_size
        self.canvas.config(width=c * cs, height=r * cs)
        self._update_status("Grid resized")
        self._redraw()

    def random_map(self):
        self._stop_all()
        try:
            d = max(0, min(100, int(self.density_var.get()))) / 100.0
        except ValueError:
            d = 0.30
        self.grid.generate_random(d)
        self._reset_metrics()
        self._update_status("Random map generated")
        self._redraw()

    def start_search(self):
        if self.searching:
            return
        self.clear_search()
        self.search_result = SearchResult()
        self.replan_count  = 0
        self.agent_pos     = None
        self.agent_moving  = False
        self.active_path   = []
        self.path_idx      = 0

        h_fn = euclidean if self.heuristic_var.get() == "Euclidean" else manhattan
        self.search_gen = search_step_generator(
            self.grid, self.algorithm.get(), h_fn
        )
        self.searching = True
        self._update_status("Searching...", "#f9e2af")
        self._step_search()

    def clear_search(self):
        self._stop_all()
        self.grid.clear_visuals()
        self._reset_metrics()
        self._update_status("Idle")
        self._redraw()

    def clear_all(self):
        self._stop_all()
        self.grid.reset()
        self._reset_metrics()
        self._update_status("Idle")
        self._redraw()

    def _reset_metrics(self):
        self.search_result = SearchResult()
        self.replan_count  = 0
        self._update_metrics()

    def _stop_all(self):
        """Cancel any running search or agent movement."""
        self.searching    = False
        self.search_gen   = None
        self.agent_moving = False
        self.active_path  = []
        self.agent_pos    = None
        if self.search_job:
            self.root.after_cancel(self.search_job)
            self.search_job = None
        if self.agent_job:
            self.root.after_cancel(self.agent_job)
            self.agent_job = None

    # ============================================================
    # SEARCH ANIMATION LOOP  (called via after())
    # ============================================================
    def _step_search(self):
        """Advance search by `speed` steps then schedule next frame."""
        if not self.searching or self.search_gen is None:
            return

        steps = self.speed_var.get()
        done  = False

        for _ in range(steps):
            try:
                frame = next(self.search_gen)
                self.search_result = frame['result']
                if frame['type'] == 'done':
                    done = True
                    break
            except StopIteration:
                done = True
                break

        self._update_metrics()
        self._redraw()

        if done:
            self.searching  = False
            self.search_gen = None
            if self.search_result.success:
                self._update_status("Path found! ✓", "#a6e3a1")
                # Start agent movement if dynamic mode is enabled
                if self.dynamic_var.get() and self.search_result.path:
                    self.active_path  = list(self.search_result.path)
                    self.path_idx     = 0
                    self.agent_pos    = self.active_path[0]
                    self.agent_moving = True
                    self.agent_job = self.root.after(AGENT_DELAY, self._step_agent)
            else:
                self._update_status("No path found ✕", "#f38ba8")
        else:
            self.search_job = self.root.after(ANIM_DELAY, self._step_search)

    # ============================================================
    # AGENT ANIMATION LOOP  (dynamic mode)
    # ============================================================
    def _step_agent(self):
        """Move agent one step, spawn obstacles, replan if needed."""
        if not self.agent_moving:
            return

        self.path_idx += 1
        if self.path_idx >= len(self.active_path):
            self.agent_moving = False
            self.agent_pos    = None
            self._update_status("Goal reached! ✓", "#a6e3a1")
            self._redraw()
            return

        self.agent_pos = self.active_path[self.path_idx]

        # ── Spawn dynamic obstacle ──
        protected = {self.agent_pos, self.grid.start, self.grid.goal}
        obstacle  = self.grid.spawn_obstacle(probability=0.08, protected=protected)

        if obstacle:
            # Efficiency check: only replan if obstacle is ON remaining path
            remaining_set = set(self.active_path[self.path_idx:])
            if obstacle in remaining_set:
                self._replan()
                return   # _replan re-schedules agent step

        self._update_metrics()
        self._redraw()
        self.agent_job = self.root.after(AGENT_DELAY, self._step_agent)

    def _replan(self):
        """Instantly recompute path from current agent position."""
        self.replan_count += 1
        self.grid.clear_visuals()
        h_fn = euclidean if self.heuristic_var.get() == "Euclidean" else manhattan
        result = run_instant_search(
            self.grid, self.algorithm.get(), h_fn,
            start_override=self.agent_pos
        )
        self.search_result = result
        if result.success and result.path:
            self.active_path  = list(result.path)
            self.path_idx     = 0
            self.agent_pos    = self.active_path[0]
            self._update_status(f"Re-planned (#{self.replan_count})", "#fab387")
        else:
            self.agent_moving = False
            self._update_status("No path after re-plan ✕", "#f38ba8")
        self._update_metrics()
        self._redraw()
        if self.agent_moving:
            self.agent_job = self.root.after(AGENT_DELAY, self._step_agent)

    # ============================================================
    # MOUSE INTERACTION
    # ============================================================
    def _cell_from_event(self, event):
        cs = self.cell_size
        c  = event.x // cs
        r  = event.y // cs
        if 0 <= r < self.grid.rows and 0 <= c < self.grid.cols:
            return (r, c)
        return None

    def _on_click(self, event):
        cell = self._cell_from_event(event)
        if not cell:
            return
        r, c = cell

        if self.draw_mode == "start":
            self.grid.move_start(r, c)
            self.draw_mode = "wall"
            self._update_status("Start placed")
        elif self.draw_mode == "goal":
            self.grid.move_goal(r, c)
            self.draw_mode = "wall"
            self._update_status("Goal placed")
        else:
            # Toggle wall
            self.grid.toggle_wall(r, c)
            self.last_drawn_cell = cell

        self._redraw()

    def _on_drag(self, event):
        """Drag to draw/erase walls."""
        if self.draw_mode not in ("wall",):
            return
        cell = self._cell_from_event(event)
        if not cell or cell == self.last_drawn_cell:
            return
        if cell != self.grid.start and cell != self.grid.goal:
            self.grid.set_wall(*cell)
            self.last_drawn_cell = cell
            self._redraw()

    def _on_right_click(self, event):
        """Right-click to erase a wall."""
        cell = self._cell_from_event(event)
        if cell:
            self.grid.remove_wall(*cell)
            self._redraw()

    # ============================================================
    # RUN
    # ============================================================
    def run(self):
        self.root.mainloop()


# ==============================================================
# ENTRY POINT
# ==============================================================
if __name__ == "__main__":
    app = PathfindingApp()
    app.run()
